In [1]:
from collections import OrderedDict
import itertools as it

In [2]:
type SubstrMap = dict[str, tuple[int, list[tuple[int, int]]]]

In [3]:
def get_substr(s: str) -> SubstrMap:
    """Get non-overlapping substrings from a string with its value count and inclusion positions"""
    pr = OrderedDict()
    all_subs = [] # should be an orderedset
    for l in range(1, len(s)):
        for start in range(0, len(s) - 1):
            # Get all possible substrings
            sub = s[start:start + l]
            if sub not in all_subs:
                all_subs.append(sub)

    for sub in all_subs:
        # scan for the positions:
        start = 0
        while start < (len(s) - len(sub) + 1):
            if s[start:start + len(sub)] == sub:
                if pr.get(sub) is not None:
                    old = pr[sub]
                    pr[sub] = (old[0] + 1, old[1] + [(start, start + len(sub))])
                else:
                    pr[sub] = (1, [(start, start + len(sub))])
                start += len(sub) # skip over
            else:
                start += 1
    return pr


In [4]:
def mut_strong(pr: SubstrMap) -> tuple[dict[str, list[str]], dict[str, list[str]]]:
    """Find strongly mutually exclusive substrings, returns the s2->{s1} and s1->{s2} mappings and the {s1, s2} pairs"""
    mut = OrderedDict()  # cannot contain: s2 -> s1
    cbi = OrderedDict()  # cannot be in: s1 -> s2
    pairs = []
    for i, (s1, (count1, pos1)) in enumerate(pr.items()):
        for j, (s2, (count2, pos2)) in enumerate(pr.items()):
            
            # strong condition: # of inclusions must be the same
            if j == i or (not s1 in s2) or len(pos1) == 0 or len(pos1) != len(pos2):
                continue
            # also skip subs of size 1
            #if count1 == 1 or count2 == 1:
            #    continue
            
            # Check strong condition
            # We assume that sls is sorted
            for k in range(len(pos2)):
                if not (pos1[k][0] >= pos2[k][0] and pos1[k][1] <= pos2[k][1]):
                    continue
            pairs.append((s1, s2))
            if mut.get(s2) is not None:
                mut[s2].append(s1)
            else:
                mut[s2] = [s1]
            if cbi.get(s1) is not None:
                cbi[s1].append(s2)
            else:
                cbi[s1] = [s2]
    return mut, cbi, pairs


In [5]:
s = "1111[11[1[0]0]1[0]0]11[1[0]0]1[0]0"

In [6]:
pr = get_substr(s)

In [8]:
#pr

## Filter out

In [9]:
len(pr)  # All substrings

441

In [10]:
len([x for x in pr if len(x) > len(s) / 2])  # Substrings which are larger than half of the string

152

In [11]:
len([x for x in pr if len(x) == 1])  # Unit substrings

4

In [12]:
len([x for x in pr.values() if x[0] == 1])  # Substrings which occur only once
# Use this as a decent-enough heuristic for long enough recursion depths

358

In [13]:
pr_f = OrderedDict([(k, v) for k, v in pr.items() if v[0] > 1 and len(k) > 1])
#pr_f

In [14]:
# Find mutually exclusive substrings
mut, cbi, pairs = mut_strong(pr_f)

In [15]:
len(pairs)  # number of such pairs

735

# Gene mapping algorithms

## Count (naive)

In [16]:
list(pr_f.items())[0]

('11', (4, [(0, 2), (2, 4), (5, 7), (20, 22)]))

In [43]:
groups = []

get_count = lambda x: x[1][0]
data = sorted(pr_f.items(), key=get_count)  # is stable
for key, group in it.groupby(data, get_count):
    groups.append((key, OrderedDict(group)))

In [44]:
groups

[(2,
  OrderedDict([('[1[', (2, [(7, 10), (22, 25)])),
               (']1[', (2, [(13, 16), (28, 31)])),
               ('11[1', (2, [(2, 6), (20, 24)])),
               ('1[1[', (2, [(6, 10), (21, 25)])),
               ('[1[0', (2, [(7, 11), (22, 26)])),
               ('0]1[', (2, [(12, 16), (27, 31)])),
               (']1[0', (2, [(13, 17), (28, 32)])),
               ('11[1[', (2, [(5, 10), (20, 25)])),
               ('1[1[0', (2, [(6, 11), (21, 26)])),
               ('[1[0]', (2, [(7, 12), (22, 27)])),
               (']0]1[', (2, [(11, 16), (26, 31)])),
               ('0]1[0', (2, [(12, 17), (27, 32)])),
               (']1[0]', (2, [(13, 18), (28, 33)])),
               ('11[1[0', (2, [(5, 11), (20, 26)])),
               ('1[1[0]', (2, [(6, 12), (21, 27)])),
               ('[1[0]0', (2, [(7, 13), (22, 28)])),
               ('0]0]1[', (2, [(10, 16), (25, 31)])),
               (']0]1[0', (2, [(11, 17), (26, 32)])),
               ('0]1[0]', (2, [(12, 18), (27, 33)])),
  

## Lex

In [20]:
pr_lex = OrderedDict(sorted(pr_f.items(), key=lambda x: x[0]))

In [46]:
groups_lex = []
cur_len = 0

for k, v in pr_lex.items():
    if cur_len != 0 and len(k) > cur_len:
        groups_lex[-1][1][k] = v
    else:
        # create new group
        groups_lex.append((0, OrderedDict([(k, v)])))
    cur_len = len(k)

groups_lex

[(0,
  OrderedDict([('0]',
                (7,
                 [(10, 12),
                  (12, 14),
                  (16, 18),
                  (18, 20),
                  (25, 27),
                  (27, 29),
                  (31, 33)])),
               ('0]0', (4, [(10, 13), (16, 19), (25, 28), (31, 34)])),
               ('0]0]', (3, [(10, 14), (16, 20), (25, 29)])),
               ('0]0]1', (3, [(10, 15), (16, 21), (25, 30)])),
               ('0]0]1[', (2, [(10, 16), (25, 31)])),
               ('0]0]1[0', (2, [(10, 17), (25, 32)])),
               ('0]0]1[0]', (2, [(10, 18), (25, 33)])),
               ('0]0]1[0]0', (2, [(10, 19), (25, 34)]))])),
 (0,
  OrderedDict([('0]1', (3, [(12, 15), (18, 21), (27, 30)])),
               ('0]1[', (2, [(12, 16), (27, 31)])),
               ('0]1[0', (2, [(12, 17), (27, 32)])),
               ('0]1[0]', (2, [(12, 18), (27, 33)])),
               ('0]1[0]0', (2, [(12, 19), (27, 34)]))])),
 (0,
  OrderedDict([('11', (4, [(0, 2), (2, 4), (5

## Seed

In [41]:
def seeds_merged(cbi: dict[str, list[str]]) -> tuple[dict[str, int], int]:
    """
    Get seeds from mutually exclusive strings, returns the substr->seed mapping and the largest unused seed (not len(seeds)).
    Seeds of {s1}->s2 are merged: all seed branches a->b->c, d->b->c are flattened into one seed, elements a,d are not mutex.
    """
    # Get all seeds
    s1_seeds = OrderedDict()
    
    next_seed = 0
    for s1, s2_set in cbi.items():
        # check if the seed already exists for s1 and {s2}
        cur_seeds = set()
        if s1 in s1_seeds:
            cur_seeds.add(s1_seeds[s1])
        # We still have to check the seeds of {s2} in case if there are overlapping seed groups
        for s2 in s2_set:
            if s2 in s1_seeds:  # s2 may have a larger substring counterpart
                cur_seeds.add(s1_seeds[s2])
    
        if len(cur_seeds) == 0:
            # New seed
            s1_seeds[s1] = next_seed
            for s2 in s2_set:
                s1_seeds[s2] = next_seed
            next_seed += 1
        elif len(cur_seeds) == 1:
            # apply the existing seed
            new_seed = list(cur_seeds)[0]
            s1_seeds[s1] = new_seed
            for s2 in s2_set:
                s1_seeds[s2] = new_seed
        else:
            # seed groups intersect, need to merge
            print(f"seed groups intersect : {cur_seeds}")
            new_seed = list(cur_seeds)[0]
            # O(n) update
            for k in s1_seeds:
                if s1_seeds[k] in cur_seeds:
                    s1_seeds[k] = new_seed
            
            s1_seeds[s1] = new_seed
            for s2 in s2_set:
                s1_seeds[s2] = new_seed

    # TODO reindex seeds to be strictly sequential
    return s1_seeds, next_seed

In [42]:
def strong_seeds(cbi: dict[str, list[str]]) -> tuple[dict[str, int], int]:
    """
    Get strong seeds from mutually exclusive strings, returns the substr->seed mapping and the largest unused seed (not len(seeds)).
    Preserves mutability: each branch s1->s2->s3 has a distinct seed.
    """
    pass

In [39]:
s1_seeds = seeds_merged(cbi)
#s1_seeds

In [24]:
[x for x in pr_f if x not in s1_seeds]  # substr without a seed

['11', '1[', '0]', '11[']

In [28]:
pr_seeds = OrderedDict()

# Add seed to the substrings
next_seed = num_seeds
for k, v in pr_f.items():
    if k in s1_seeds:
        cur_seed = s1_seeds[k]
    else:
        cur_seed = next_seed
        next_seed += 1
        
    pr_seeds[k] = (v[0], cur_seed, v[1])

In [27]:
#pr_seeds

In [48]:
# Group by seed
groups_seed = []

get_seed = lambda x: x[1][1]
data = sorted(pr_seeds.items(), key=get_seed)  # is stable
for key, group in it.groupby(data, get_seed):
    # Sort by lex in the seed group
    group_lex = sorted(map(lambda y: (y[0], (y[1][0], y[1][2])), group), key=lambda x: x[0])
    groups_seed.append((key, OrderedDict(group_lex)))
# Sort groups by seed 

In [49]:
groups_seed

[(0,
  OrderedDict([('1[1', (3, [(3, 6), (6, 9), (21, 24)])),
               ('[1', (3, [(4, 6), (7, 9), (22, 24)]))])),
 (1,
  OrderedDict([('0]0', (4, [(10, 13), (16, 19), (25, 28), (31, 34)])),
               ('1[0', (4, [(8, 11), (14, 17), (23, 26), (29, 32)])),
               ('1[0]', (4, [(8, 12), (14, 18), (23, 27), (29, 33)])),
               ('1[0]0', (4, [(8, 13), (14, 19), (23, 28), (29, 34)])),
               ('[0', (4, [(9, 11), (15, 17), (24, 26), (30, 32)])),
               ('[0]', (4, [(9, 12), (15, 18), (24, 27), (30, 33)])),
               ('[0]0', (4, [(9, 13), (15, 19), (24, 28), (30, 34)])),
               (']0', (4, [(11, 13), (17, 19), (26, 28), (32, 34)]))])),
 (2,
  OrderedDict([('0]0]', (3, [(10, 14), (16, 20), (25, 29)])),
               ('0]0]1', (3, [(10, 15), (16, 21), (25, 30)])),
               ('0]1', (3, [(12, 15), (18, 21), (27, 30)])),
               ('1[0]0]', (3, [(8, 14), (14, 20), (23, 29)])),
               ('[0]0]', (3, [(9, 14), (15, 20), (24,

In [ ]:
# Check seeds for mutex
for g in groups_seed:
    print(g[0])
    for i, (k, v) in enumerate(g[1]):
        for j, (k2, v2) in enumerate(g[1]):
            if j <= i:
                continue
            for l in range(len(v[1])):
                if not (v[1][l][0] >= v2[1][l][0] and v[1][l][1] <= v2[1][l][1]):
                    print(f"not mutex : {k}, {k2}")
                    break